# Electricity Access Analysis for Sub-Saharan Africa

## Objective

The objective of this notebook is to analyze electricity electricity across Sub-Saharan African countries using World Bank data. The analysis explores historical trends, compares country performance, and develops a simple weighted priority model to identify countries that may warrant further electricity investment.

## Dataset

- Source: World Bank – World Development Indicators
- Indicator: Access to electricity (% of population)
- Years analyzed: 2014–2023

## Workflow

1. Load and inspect the data
2. Clean and prepare the dataset
3. Merge country metadata
4. Filter Sub-Saharan African countries
5. Perform exploratory data analysis (EDA)
6. Engineer new features
7. Build a weighted priority model
8. Perform sensitivity analysis
9. Visualize results

### project setup

In [ ]:

from    pathlib import Path
import sys

# Add the project root to Python's path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



###  Import Libraries

In [ ]:
import pandas as pd

from src.load_data import (
    load_worldbank_data,
    load_metadata,
)

from src.clean_data import (
    merge_metadata,
    filter_ssa,
    select_years,
)

from src.scoring import (
    calculate_access_improvement,
    calculate_relative_improvement,
    calculate_people_without_electricity,
    calculate_access_score,
    calculate_improvement_score,
    calculate_population_score,
    calculate_priority_score,
    calculate_priority_score_with_population,
)

from src.visualization import (
    plot_average_access,
    plot_improvement_scatter,
    plot_priority_ranking,
)

   ### Load data

In [ ]:
electricity_df = load_worldbank_data(
    "../data/raw/electricity/API_EG.ELC.ACCS.ZS_DS2_en_csv_v2_3606.csv"
)

metadata = load_metadata(
    "../data/raw/electricity/Metadata_Country_API_EG.ELC.ACCS.ZS_DS2_en_csv_v2_3606.csv"
)

### Inspect the dataset

In [ ]:
# Number of rows and columns
print("Shape:", electricity_df.shape)

# Column names
print("\nColumns:")
print(electricity_df.columns.tolist())

In [ ]:
electricity_df.info()

In [ ]:
electricity_df.head()

In [ ]:
# Count missing values in each column
electricity_df.isna().sum()

In [ ]:
electricity_df[electricity_df["2023"].isna()][["Country Name", "Country Code", "2023"]]

In [ ]:
# Show all country names
electricity_df["Country Name"]

### Clean Data

In [ ]:
# Display unique country names alphabetically
sorted(electricity_df["Country Name"].unique())

In [ ]:
metadata = load_metadata(
    "../data/raw/electricity/Metadata_Country_API_EG.ELC.ACCS.ZS_DS2_en_csv_v2_3606.csv"
)
metadata.head()

####    Merge Metadata

In [ ]:
# Merge the electricity data with the country metadata
electricity_df_merged = merge_metadata(
    electricity_df,
    metadata
)

electricity_df_merged.head()

####    Filter Sub-Saharan Africa

In [ ]:
ssa_df = filter_ssa(electricity_df_merged)
ssa_df.shape

### Exploratory Data Analysis


In [ ]:
# Select only the years we want
years = select_years()

# Summary statistics
ssa_df[years].describe().round(2)

### Vizualization

This section calculates the average electricity electricity across all 48 Sub-Saharan African countries and visualizes the trend over time (2014–2023)

In [ ]:
import matplotlib.pyplot as plt

# Calculate the average electricity electricity for each year
mean_electricity = ssa_df[years].mean()

plt.figure(figsize=(10, 5))
plt.plot(mean_electricity.index, mean_electricity.values, marker="o")

plt.title("Average Electricity Access in Sub-Saharan Africa (2014–2023)")
plt.xlabel("Year")
plt.ylabel("Access to Electricity (%)")

plt.xticks(rotation=45)
plt.grid(True)

plt.show()

### Priority Model

In [ ]:
# Countries with the highest electricity electricity in 2023
ssa_df.sort_values(by="2023", ascending=False)[
    ["Country Name", "2023"]
].head(10)

In [ ]:
ssa_df.sort_values(by="2023")[
    ["Country Name", "2023"]
].head(10)

In [ ]:
ssa_df["2023"].describe().round(2)

In [ ]:
ssa_df = calculate_access_improvement(ssa_df)

ssa_df.sort_values(
    by="Improvement",
    ascending=False
)[["Country Name", "2014", "2023", "Improvement"]].head(10)

### Feature Engineering

In [ ]:
ssa_df = calculate_relative_improvement(ssa_df)

ssa_df.sort_values(
    by="Relative Improvement (%)",
    ascending=False
)[["Country Name", "2014", "2023", "Relative Improvement (%)"]].head(10)

In [ ]:
comparison = ssa_df[["2014", "2023"]].describe().round(2)

comparison.loc[["mean", "std", "min", "25%", "50%", "75%", "max"]]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.scatter(ssa_df["2014"], ssa_df["Improvement"])

plt.xlabel("Electricity Access in 2014 (%)")
plt.ylabel("Improvement (2014–2023)")
plt.title("Starting Electricity Access vs Improvement")

plt.grid(True)

plt.show()

In [ ]:
ssa_df[["2014", "Improvement"]].corr()

In [ ]:
# Access_score is the normalized 2023 electricity electricity, we reverse it so that lower electricity access gets a higher score
ssa_df = calculate_access_score(ssa_df)

ssa_df[["Country Name", "2023", "Access_Score"]].head()

In [ ]:
# Normalize Improvement
ssa_df = calculate_improvement_score(ssa_df)
ssa_df[["Country Name", "Improvement", "Improvement_Score"]].head()

In [ ]:
ssa_df[ssa_df["Country Name"] == "Rwanda"][
    ["Country Name", "Improvement", "Improvement_Score"]
]

In [ ]:
# Calculate the priority score for equal weights
ssa_df = calculate_priority_score(
    ssa_df,
    access_weight=0.5,
    improvement_weight=0.5
)
# Show the top 10 countries
priority = (
    ssa_df[["Country Name", "2023", "Improvement", "Priority_Score"]]
    .sort_values("Priority_Score", ascending=False)
    .head(10)
)

priority

In [ ]:
import matplotlib.pyplot as plt

# Prepare data
priority = (
    ssa_df[["Country Name", "Priority_Score"]]
    .sort_values("Priority_Score", ascending=False)
    .head(10)
)

# Create figure
plt.figure(figsize=(10, 6))

# Draw bars
bars = plt.barh(priority["Country Name"], priority["Priority_Score"])

# Put highest score at the top
plt.gca().invert_yaxis()

# Add score labels to each bar
for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 0.01,
        bar.get_y() + bar.get_height()/2,
        f"{width:.2f}",
        va="center"
    )

# Titles
plt.title("Top 10 Priority Countries for Electricity Investment")
plt.xlabel("Priority Score")
plt.ylabel("Country")

# Limit x-axis for better spacing
plt.xlim(0, 1)

plt.tight_layout()
plt.show()

### Sensitivity Analysis

In [ ]:
# New weights: 80% Access, 20% Improvement

ssa_df = calculate_priority_score(
    ssa_df,
    access_weight=0.8,
    improvement_weight=0.2,
    output_column="Priority_80_20"
)

# Top 10 countries
priority_80_20 = (
    ssa_df[
        [
            "Country Name",
            "2023",
            "Improvement",
            "Priority_80_20"
        ]
    ]
    .sort_values("Priority_80_20", ascending=False)
    .head(10)
)

priority_80_20

In [ ]:
ssa_df[["Country Name", "Access_Score", "Improvement_Score", "Priority_Score"]].head(10)

### Save Clean Dataset

In [ ]:
ssa_df.to_csv(
    "../data/processed/ssa_electricity_clean_with_priority_score.csv",
    index=False
)

# Load population data
We want to know the total population without access to electricity, so we load and merge total population data to electricity access data from World bank database

In [ ]:
population_df = load_worldbank_data(
    "../data/raw/pop/API_SP.POP.TOTL_DS2_en_csv_v2_33112.csv"
)

In [ ]:
# Number of rows and columns
print("Shape:", population_df.shape)


In [ ]:
# Column names
print("\nColumns:")
print(population_df.columns.tolist())

In [ ]:
# Count missing values in each column
population_df.isna().sum()


In [ ]:
# Display unique country names alphabetically
sorted(population_df["Country Name"].unique())

In [ ]:
# Merge the population data with the country metadata
population_df = merge_metadata(
    population_df,
    metadata
)

# Rename the 2023 column to Population_2023 for clarity
population_df.rename(
    columns={"2023": "Population_2023"},
    inplace=True
)


In [ ]:

# Then filter SSA
ssapop_df = filter_ssa(population_df)

population_subset = ssapop_df[
    ["Country Code", "Population_2023"]
]

In [ ]:
# Merge population data into the electricity dataframe
ssa_df = ssa_df.merge(
    population_subset,
    on="Country Code",
    how="left"
)

# Create the People_Without_Electricity column
ssa_df = calculate_people_without_electricity(ssa_df)

# Check the result
ssa_df[
    [
        "Country Name",
        "Population_2023",
        "2023",
        "People_Without_Electricity",
    ]
].head()

In [ ]:
# Normalize People_Without_Electricity within the SSA region to create a new score column

ssa_df = calculate_population_score(ssa_df) 

In [ ]:

# call the Population-adjusted model with equal weights for access, improvement, and population
ssa_df = calculate_priority_score_with_population(
    ssa_df,
    access_weight=1/3,
    improvement_weight=1/3,
    population_weight=1/3,
)

In [ ]:
priority = (
    ssa_df[
        [
            "Country Name",
            "2023",
            "Improvement",
            "People_Without_Electricity",
            "Population_Score",
            "Priority_Score"
        ]
    ]
    .sort_values("Priority_Score", ascending=False)
    .head(10)
)

priority

Version 1 Summary

Objective

Develop a priority ranking model for Sub-Saharan African countries using electricity access, improvement in electricity access (2014–2023), and the estimated number of people without electricity.

Data Sources

World Bank: Access to Electricity
World Bank: Total Population

Methodology

Filter to SSA countries.
Calculate absolute improvement (2023 − 2014).
Normalize variables using min–max scaling.
Estimate people without electricity.
Build a weighted priority score.
Compare rankings before and after incorporating population.

Key Finding

Including population substantially changes the rankings, highlighting countries such as DR Congo and Nigeria because of the large number of people still lacking electricity.